# 02 — Relay activity (MITRE ATT&CK T1090, Proxy)

Analysis of `cowrie.direct-tcpip.request` events across the three-day sample.

**What this event is.** An SSH client asks the honeypot to open a TCP connection
to some other host on its behalf. That is the `direct-tcpip` channel type in
SSH: the client is trying to **use the honeypot as a relay**, so traffic to the
real target appears to come from the honeypot rather than from the attacker.
In MITRE ATT&CK this is **T1090 — Proxy**.

**Method constraint.** `tcpip_target` is a SHA-256 hash of the destination
address, exactly like `src_ip_identifier`. Targets therefore **cannot be
geolocated or attributed**, and nothing here attempts it. Everything below is a
statement about *patterns* — concentration, recurrence, distribution — which
survives pseudonymisation. Any claim of the form "attacker X targeted host Y"
is not supportable from this data, and the hashed prefixes shown in tables are
identifiers only, not intelligence.

**Source.** `data/interim/`, built by `scripts/build_dataset.py`. See the README
for why that folder is git-ignored and what is redacted from it.

## Setup

In [ ]:
import pathlib

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 70)


def find_interim(start=pathlib.Path.cwd()):
    '''Walk upward to the repo root so the notebook runs from either location.'''
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "interim").is_dir():
            return candidate / "data" / "interim"
    raise FileNotFoundError(
        "Could not find data/interim/. Run scripts/build_dataset.py first."
    )


INTERIM = find_interim()
print(f"reading from: {INTERIM}\n")

sessions = pd.read_parquet(INTERIM / "sessions.parquet")
events = pd.read_parquet(
    f"{INTERIM}/events.parquet",
    columns=["eventid", "session_id", "src_ip_identifier", "sensor",
             "day", "tcpip_target", "tcpip_port"],
)

relay = events[events["eventid"] == "cowrie.direct-tcpip.request"].copy()
relay = relay.merge(sessions[["session_id", "sensor", "has_login_success", "n_commands"]],
                    on="session_id", suffixes=("", "_s"))

print(f"sessions (all)      : {len(sessions):,}")
print(f"distinct attackers  : {sessions['src_ip_identifier'].nunique():,}")
print(f"relay requests      : {len(relay):,}")
print(f"relay sessions      : {relay['session_id'].nunique():,}")
print(f"relay attackers     : {relay['src_ip_identifier'].nunique():,}")
print(f"distinct targets    : {relay['tcpip_target'].nunique():,}")
print(f"distinct ports      : {relay['tcpip_port'].nunique():,}")

## 1. How concentrated is relay activity?

The first thing to establish is scale. Relay events are a tiny slice of the
corpus, and the interesting question is whether that slice is one busy attacker
or a broad population.

In [ ]:
relay_sessions = sessions[sessions["n_directtcpip"] > 0]
attacker_sessions = sessions["src_ip_identifier"].nunique()
relay_attackers = relay["src_ip_identifier"].nunique()

stats = pd.Series({
    "all sessions": len(sessions),
    "sessions that relayed": len(relay_sessions),
    "distinct attackers (all)": attacker_sessions,
    "distinct attackers that relayed": relay_attackers,
    "relay requests": len(relay),
    "distinct relay targets": relay["tcpip_target"].nunique(),
})
print(f"relay sessions   : {len(relay_sessions):,} of {len(sessions):,} "
      f"({len(relay_sessions)/len(sessions)*100:.2f}%)")
print(f"relay attackers  : {relay_attackers} of {attacker_sessions:,} "
      f"({relay_attackers/attacker_sessions*100:.2f}%)")
print()
print(f"{'relay requests per session':<32} "
      f"{relay.groupby('session_id').size().median():.0f} (median), "
      f"{relay.groupby('session_id').size().max()} (max)")
print()
stats.to_frame("value")

In [ ]:
top_sessions = relay.groupby("session_id").size().sort_values(ascending=False).head(40)
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.bar(range(len(top_sessions)), top_sessions.values, color="#c0392b")
ax.set_xlabel("40 busiest relay sessions (rank)")
ax.set_ylabel("direct-tcpip requests")
ax.set_title("Relay volume per session is extremely skewed")
plt.tight_layout()
plt.show()

## 2. Port distribution

`tcpip_port` is a genuinely useful field here: unlike the address, the port
number is **not** pseudonymised, so it can be read directly.

In [ ]:
ports = relay["tcpip_port"].value_counts().rename("requests").to_frame()
ports["pct"] = (ports["requests"] / len(relay) * 100).round(2)
print(f"{len(ports)} distinct ports across {len(relay):,} requests\n")
ports

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(ports.index.astype(int).astype(str), ports["requests"], color="#2c3e50")
ax.set_xlabel("target port")
ax.set_ylabel("requests")
ax.set_title("Relay targets: port distribution (port 80 dominates)")
plt.tight_layout()
plt.show()

## 3. Targets per attacker

For each relaying attacker, how many distinct relay targets did it try to reach?

In [ ]:
per_attacker = relay.groupby("src_ip_identifier").agg(
    sessions=("session_id", "nunique"),
    targets=("tcpip_target", "nunique"),
    requests=("tcpip_target", "size"),
    days=("day", "nunique"),
)

print(f"{len(per_attacker)} relaying attackers\n")
print(per_attacker.describe().round(2).to_string())

In [ ]:
top = per_attacker.sort_values("requests", ascending=False).head(15)
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.bar(range(len(top)), top["requests"], color="#8e44ad")
ax.set_xlabel("attacker (rank by relay requests)")
ax.set_ylabel("requests")
ax.set_title("Top 15 relaying attackers by request volume")
plt.xticks(range(len(top)), [str(i + 1) for i in range(len(top))])
plt.tight_layout()
plt.show()

print(f"top 1 attacker  : {top['requests'].iloc[0]:,} requests "
      f"({top['requests'].iloc[0]/len(relay)*100:.1f}% of all relay)")
print(f"top 5 attackers : {top['requests'].head(5).sum():,} requests "
      f"({top['requests'].head(5).sum()/len(relay)*100:.1f}%)")

## 4. Sessions per attacker IP

How many sessions did each relaying attacker open?

In [ ]:
s = per_attacker["sessions"]
print("sessions per relaying attacker:")
print(f"  median {s.median():.0f}   mean {s.mean():.1f}   max {s.max()}")
print(f"  attackers with exactly 1 session : {int((s == 1).sum())}")
print(f"  attackers with > 100 sessions    : {int((s > 100).sum())}\n")
s.value_counts().sort_index().head(12).rename("attackers").to_frame()

## 5. Repeat activity across the three days

Did the same attacker persist across the sample window, or is each day's relay
activity a separate population?

In [ ]:
days_active = per_attacker["days"].value_counts().sort_index()
days_active.rename("attackers").to_frame().assign(
    pct=lambda d: (d["attackers"] / len(per_attacker) * 100).round(1)
)

In [ ]:
per_day = relay.groupby("day", observed=True).agg(
    requests=("session_id", "size"),
    sessions=("session_id", "nunique"),
    attackers=("src_ip_identifier", "nunique"),
    targets=("tcpip_target", "nunique"),
)
per_day["requests_per_attacker"] = (per_day["requests"] / per_day["attackers"]).round(1)
print(per_day.to_string())
print(f"\nattackers seen on more than one day: "
      f"{int((per_attacker['days'] > 1).sum())} of {len(per_attacker)} "
      f"({(per_attacker['days'] > 1).mean()*100:.0f}%)")

## 6. Split by sensor

The sample is not one experiment configuration (see the README scope section).
`ubuntu_basic_pool` is the high-interaction mode; the four `prod-*-local-*`
nodes are a different configuration. **A difference between them is not
automatically a difference in attacker behaviour** — the honeypot mode changes
what an attacker can observe and do.

In [ ]:
by_sensor = relay.groupby("sensor_s", observed=True).agg(
    requests=("session_id", "size"),
    sessions=("session_id", "nunique"),
    attackers=("src_ip_identifier", "nunique"),
    targets=("tcpip_target", "nunique"),
    ports=("tcpip_port", "nunique"),
)
by_sensor["pct_of_requests"] = (by_sensor["requests"] / len(relay) * 100).round(2)
by_sensor["requests_per_attacker"] = (by_sensor["requests"] / by_sensor["attackers"]).round(1)
print(by_sensor.sort_values("requests", ascending=False).to_string())

shared = relay.groupby("tcpip_target", observed=True)["sensor_s"].nunique()
print(f"\ntargets contacted by more than one sensor: "
      f"{int((shared > 1).sum())} of {len(shared)}")

In [ ]:
order = by_sensor.sort_values("requests", ascending=False)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(order.index.astype(str), order["requests"], color="#16a085")
ax.set_ylabel("relay requests")
ax.set_title("Relay requests by honeypot configuration")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 7. Two structural findings

These are the parts worth putting in the report, because they describe
*structure* rather than volume.

### 7a. Relaying requires a successful login

Every relay session is a subset of the sessions that successfully authenticated.

In [ ]:
all_sessions_with_success = int(sessions["has_login_success"].sum())
print(f"sessions with login.success      : {all_sessions_with_success:,} of {len(sessions):,} "
      f"({all_sessions_with_success/len(sessions)*100:.2f}%)")
print(f"relay sessions                   : {len(relay_sessions):,}")
print(f"relay sessions with login.success: {int(relay_sessions['has_login_success'].sum()):,} "
      f"({relay_sessions['has_login_success'].mean()*100:.1f}%)")

cmd_sessions = set(events.loc[events["eventid"] == "cowrie.command.input", "session_id"])
relay_ids = set(relay["session_id"])
print(f"\ninteractions = {len(cmd_sessions):,} sessions   "
      f"relay = {len(relay_ids):,} sessions   overlap = {len(cmd_sessions & relay_ids)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
rates = pd.Series({
    "all sessions": sessions["has_login_success"].mean() * 100,
    "sessions with commands": sessions.loc[sessions["n_commands"] > 0, "has_login_success"].mean() * 100,
    "relay sessions": relay_sessions["has_login_success"].mean() * 100,
})
ax.barh(rates.index, rates.values, color=["#7f8c8d", "#2980b9", "#c0392b"])
ax.set_xlabel("% of sessions reaching login.success")
ax.set_title("Relay is not opportunistic: it only happens after a successful login")
for i, v in enumerate(rates.values):
    ax.text(v + 1, i, f"{v:.1f}%", va="center")
plt.tight_layout()
plt.show()

### 7b. Relay and interactive commands never co-occur

Not one session contains both a `command.input` event and a
`direct-tcpip.request` event. The two populations are **completely disjoint**.

In [ ]:
print(f"sessions with commands : {len(cmd_sessions):,}")
print(f"sessions with relay    : {len(relay_ids):,}")
print(f"sessions with both     : {len(cmd_sessions & relay_ids):,}")
print(f"sessions with neither  : {len(sessions) - len(cmd_sessions | relay_ids):,}")

## Limits of this section

State these in the report rather than letting a reader assume otherwise.

1. **Targets cannot be identified.** `tcpip_target` is a SHA-256 hash. Nothing
   here supports naming a victim host, geolocating one, or attributing an
   attack to a named campaign.
2. **Three days is a small window.** Repeat-activity figures describe this
   sample, not the campaign's lifetime.
3. **Five sensor configurations share the sample.** Any cross-sensor comparison
   is confounded with honeypot mode.
4. **T1090 is pinned at the technique level.** Sub-techniques (T1090.001
   Internal Proxy vs T1090.002 External Proxy) depend on the attacker's own
   topology, which this data cannot reveal. Do not assert a sub-technique.
5. **Absence of commands may reflect the honeypot, not the attacker.** A relay
   request opens a channel but carries no shell output, so Cowrie may simply
   not record interaction on that path. The disjointness is real in the data;
   its interpretation as "attackers did nothing else" is not established.

## What next

- Cross-reference relay attackers against `hassh` to see whether relaying is a
  property of specific malware rather than of specific IPs.
- Compare relay-session duration against the session population.
- Extract the top relay targets *inside the git-ignored interim layer* for your
  own investigation, keeping them out of anything committed.